In [1]:
# Load the posts data (already filtered to the first two weeks)
import pandas as pd
import pyarrow.parquet as pq

# Plan 
My fianl goal is to create a table where each row is an user id, and I have columns such as:
1) Num like day one. 
2) Num posts day two.
3) Num of blocks day three. 
4) Joining date.

# ToDo
1) Add some rows with the timing, such as timestamp of first post. 

# Grouping by day

In [2]:
posts_path = "../data/posting/filtered/chunk_0_posts.parquet"
posts_table = pq.read_table(posts_path)
posts_df = posts_table.to_pandas()

print(f"Initial posts count: {len(posts_df)}")
print(f"Unique users: {posts_df['did_id'].nunique()}")

# Load join dates (saved earlier) and normalize column name to 'join_date'
join_dates_path = "../data/posting/processed/join_dates.parquet"
join_dates_df = pd.read_parquet(join_dates_path)
if 'created_at' in join_dates_df.columns:
    join_dates_df = join_dates_df.rename(columns={'created_at': 'join_date'})

# Prepare a base user table (one row per user) and attach join_date
user_table = pd.DataFrame({'did_id': posts_df['did_id'].unique()})
user_table = user_table.merge(join_dates_df[['did_id', 'join_date']], on='did_id', how='left')

# Ensure datetime types
posts_df['created_at'] = pd.to_datetime(posts_df['created_at'])
user_table['join_date'] = pd.to_datetime(user_table['join_date'])

# Merge posts with join dates so each post has its user's join_date
merged = posts_df.merge(user_table[['did_id', 'join_date']], on='did_id', how='left')

# Compute days since join (integer days) and keep only first 7 days (0..6)
merged['days_since_join'] = ((merged['created_at'] - merged['join_date']).dt.total_seconds() / (24 * 3600)).round().astype('Int64')
first_week_posts = merged[(merged['days_since_join'].notna()) & (merged['days_since_join'] >= 0) & (merged['days_since_join'] <= 6)].copy()

# Aggregate counts per user per day and produce a length-7 vector column
posts_counts = first_week_posts.groupby(['did_id', 'days_since_join']).size().unstack(fill_value=0)
for d in range(7):
    if d not in posts_counts.columns:
        posts_counts[d] = 0
posts_counts = posts_counts[sorted(posts_counts.columns)]
posts_counts = posts_counts.reset_index()
posts_counts['posts_vec'] = posts_counts[[d for d in range(7)]].values.tolist()
posts_vec_df = posts_counts[['did_id', 'posts_vec']]

# Merge posts vector into user_table
user_table = user_table.merge(posts_vec_df, on='did_id', how='left')
user_table['posts_vec'] = user_table['posts_vec'].apply(lambda x: x if isinstance(x, list) else [0]*7)

print(user_table[['did_id','posts_vec']].head())

Initial posts count: 492705
Unique users: 48625
     did_id                posts_vec
0  13255698    [0, 0, 1, 0, 0, 0, 0]
1  27401584    [4, 6, 0, 0, 0, 0, 0]
2  12123752    [0, 0, 0, 1, 0, 0, 0]
3   9037590  [1, 0, 0, 0, 17, 6, 11]
4  32928599    [1, 0, 0, 0, 0, 2, 0]
     did_id                posts_vec
0  13255698    [0, 0, 1, 0, 0, 0, 0]
1  27401584    [4, 6, 0, 0, 0, 0, 0]
2  12123752    [0, 0, 0, 1, 0, 0, 0]
3   9037590  [1, 0, 0, 0, 17, 6, 11]
4  32928599    [1, 0, 0, 0, 0, 2, 0]


In [3]:
# Blocks: produce length-7 vector columns for actor (did_id) and subject (subject_id)
blocks_path = "../data/posting/filtered/blocks.parquet"
blocks_table = pq.read_table(blocks_path)
blocks_df = blocks_table.to_pandas()

print(f"Initial blocks count: {len(blocks_df)}")

# Ensure datetime types
blocks_df['created_at'] = pd.to_datetime(blocks_df['created_at'])

# --- Actor vectors (did_id) ---
# Merge actor join_date from join_dates_df (contains did_id, join_date)
blocks_actor = blocks_df.merge(join_dates_df[['did_id', 'join_date']], on='did_id', how='left')
blocks_actor['days_since_join'] = ((blocks_actor['created_at'] - pd.to_datetime(blocks_actor['join_date'])).dt.total_seconds() / (24 * 3600)).round().astype('Int64')
blocks_actor = blocks_actor[(blocks_actor['days_since_join'].notna()) & (blocks_actor['days_since_join'] >= 0) & (blocks_actor['days_since_join'] <= 6)].copy()

# Pivot counts into columns 0..6, fill missing with 0
actor_counts = blocks_actor.groupby(['did_id', 'days_since_join']).size().unstack(fill_value=0)
for d in range(7):
    if d not in actor_counts.columns:
        actor_counts[d] = 0
actor_counts = actor_counts[sorted(actor_counts.columns)]
actor_counts = actor_counts.reset_index()
# Create vector column (list of length 7)
actor_counts['blocks_actor_vec'] = actor_counts[[d for d in range(7)]].values.tolist()
actor_vec_df = actor_counts[['did_id', 'blocks_actor_vec']]

# Merge actor vectors into user_table
user_table = user_table.merge(actor_vec_df, on='did_id', how='left')
user_table['blocks_actor_vec'] = user_table['blocks_actor_vec'].apply(lambda x: x if isinstance(x, list) else [0]*7)

# --- Subject vectors (subject_id: user being blocked) ---
# Merge subject join_date (join_dates_df joined on subject_id -> did_id)
blocks_subject = blocks_df.merge(join_dates_df.rename(columns={'did_id': 'subject_id', 'join_date': 'subject_join_date'}), on='subject_id', how='left')
blocks_subject['days_since_join_subject'] = ((blocks_subject['created_at'] - pd.to_datetime(blocks_subject['subject_join_date'])).dt.total_seconds() / (24 * 3600)).round().astype('Int64')
blocks_subject = blocks_subject[(blocks_subject['days_since_join_subject'].notna()) & (blocks_subject['days_since_join_subject'] >= 0) & (blocks_subject['days_since_join_subject'] <= 6)].copy()

# Group by subject_id and day
subject_counts = blocks_subject.groupby(['subject_id', 'days_since_join_subject']).size().unstack(fill_value=0)
for d in range(7):
    if d not in subject_counts.columns:
        subject_counts[d] = 0
subject_counts = subject_counts[sorted(subject_counts.columns)]
subject_counts = subject_counts.reset_index().rename(columns={'subject_id': 'did_id'})
subject_counts['blocks_subject_vec'] = subject_counts[[d for d in range(7)]].values.tolist()
subject_vec_df = subject_counts[['did_id', 'blocks_subject_vec']]

# Merge subject vectors into user_table
user_table = user_table.merge(subject_vec_df, on='did_id', how='left')
user_table['blocks_subject_vec'] = user_table['blocks_subject_vec'].apply(lambda x: x if isinstance(x, list) else [0]*7)

print(user_table[['did_id','blocks_actor_vec','blocks_subject_vec']].head())

Initial blocks count: 267590
     did_id        blocks_actor_vec     blocks_subject_vec
0  13255698   [0, 0, 0, 0, 2, 0, 2]  [0, 0, 1, 0, 1, 0, 0]
1  27401584   [2, 1, 0, 0, 0, 1, 0]  [2, 1, 0, 0, 0, 0, 0]
2  12123752   [0, 0, 0, 0, 0, 0, 0]  [0, 0, 0, 0, 0, 0, 0]
3   9037590  [0, 0, 0, 0, 17, 3, 2]  [0, 0, 0, 0, 0, 0, 0]
4  32928599   [0, 0, 0, 0, 0, 0, 0]  [0, 0, 0, 0, 0, 0, 0]
     did_id        blocks_actor_vec     blocks_subject_vec
0  13255698   [0, 0, 0, 0, 2, 0, 2]  [0, 0, 1, 0, 1, 0, 0]
1  27401584   [2, 1, 0, 0, 0, 1, 0]  [2, 1, 0, 0, 0, 0, 0]
2  12123752   [0, 0, 0, 0, 0, 0, 0]  [0, 0, 0, 0, 0, 0, 0]
3   9037590  [0, 0, 0, 0, 17, 3, 2]  [0, 0, 0, 0, 0, 0, 0]
4  32928599   [0, 0, 0, 0, 0, 0, 0]  [0, 0, 0, 0, 0, 0, 0]


In [4]:
# Follows: produce length-7 vector columns for follower (did_id) and followed (subject_id)
follows_path = "../data/posting/filtered/follows.parquet"
follows_table = pq.read_table(follows_path)
follows_df = follows_table.to_pandas()

print(f"Initial follows count: {len(follows_df)}")

# Ensure datetime types
follows_df['created_at'] = pd.to_datetime(follows_df['created_at'])

# --- Actor vectors (did_id as follower) ---
follows_actor = follows_df.merge(join_dates_df[['did_id','join_date']], on='did_id', how='left')
follows_actor['days_since_join'] = ((follows_actor['created_at'] - pd.to_datetime(follows_actor['join_date'])).dt.total_seconds() / (24 * 3600)).round().astype('Int64')
follows_actor = follows_actor[(follows_actor['days_since_join'].notna()) & (follows_actor['days_since_join'] >= 0) & (follows_actor['days_since_join'] <= 6)].copy()
actor_counts = follows_actor.groupby(['did_id', 'days_since_join']).size().unstack(fill_value=0)
for d in range(7):
    if d not in actor_counts.columns:
        actor_counts[d] = 0
actor_counts = actor_counts[sorted(actor_counts.columns)]
actor_counts = actor_counts.reset_index()
actor_counts['follows_actor_vec'] = actor_counts[[d for d in range(7)]].values.tolist()
actor_vec_df = actor_counts[['did_id', 'follows_actor_vec']]
user_table = user_table.merge(actor_vec_df, on='did_id', how='left')
user_table['follows_actor_vec'] = user_table['follows_actor_vec'].apply(lambda x: x if isinstance(x, list) else [0]*7)

# --- Subject vectors (subject_id: user being followed) ---
follows_subject = follows_df.merge(join_dates_df.rename(columns={'did_id': 'subject_id', 'join_date': 'subject_join_date'}), on='subject_id', how='left')
follows_subject['days_since_join_subject'] = ((follows_subject['created_at'] - pd.to_datetime(follows_subject['subject_join_date'])).dt.total_seconds() / (24 * 3600)).round().astype('Int64')
follows_subject = follows_subject[(follows_subject['days_since_join_subject'].notna()) & (follows_subject['days_since_join_subject'] >= 0) & (follows_subject['days_since_join_subject'] <= 6)].copy()
subject_counts = follows_subject.groupby(['subject_id', 'days_since_join_subject']).size().unstack(fill_value=0)
for d in range(7):
    if d not in subject_counts.columns:
        subject_counts[d] = 0
subject_counts = subject_counts[sorted(subject_counts.columns)]
subject_counts = subject_counts.reset_index().rename(columns={'subject_id': 'did_id'})
subject_counts['follows_subject_vec'] = subject_counts[[d for d in range(7)]].values.tolist()
subject_vec_df = subject_counts[['did_id', 'follows_subject_vec']]
user_table = user_table.merge(subject_vec_df, on='did_id', how='left')
user_table['follows_subject_vec'] = user_table['follows_subject_vec'].apply(lambda x: x if isinstance(x, list) else [0]*7)

print(user_table[['did_id','follows_actor_vec','follows_subject_vec']].head())

Initial follows count: 11330825
     did_id              follows_actor_vec             follows_subject_vec
0  13255698  [71, 58, 67, 119, 32, 8, 146]      [8, 18, 15, 42, 21, 3, 65]
1  27401584  [990, 369, 80, 78, 46, 28, 1]  [534, 340, 90, 66, 43, 30, 21]
2  12123752          [1, 0, 0, 1, 0, 0, 0]           [1, 0, 0, 0, 0, 0, 0]
3   9037590        [16, 0, 0, 0, 53, 6, 8]        [6, 1, 0, 0, 28, 19, 10]
4  32928599         [14, 0, 0, 0, 2, 0, 0]           [9, 5, 3, 1, 0, 0, 0]
     did_id              follows_actor_vec             follows_subject_vec
0  13255698  [71, 58, 67, 119, 32, 8, 146]      [8, 18, 15, 42, 21, 3, 65]
1  27401584  [990, 369, 80, 78, 46, 28, 1]  [534, 340, 90, 66, 43, 30, 21]
2  12123752          [1, 0, 0, 1, 0, 0, 0]           [1, 0, 0, 0, 0, 0, 0]
3   9037590        [16, 0, 0, 0, 53, 6, 8]        [6, 1, 0, 0, 28, 19, 10]
4  32928599         [14, 0, 0, 0, 2, 0, 0]           [9, 5, 3, 1, 0, 0, 0]


In [5]:
print(user_table.__len__)

<bound method DataFrame.__len__ of          did_id                        join_date                posts_vec  \
0      13255698 2024-11-11 21:30:41.308000+00:00    [0, 0, 1, 0, 0, 0, 0]   
1      27401584 2025-04-03 09:57:32.751000+00:00    [4, 6, 0, 0, 0, 0, 0]   
2      12123752 2024-08-31 02:47:56.130000+00:00    [0, 0, 0, 1, 0, 0, 0]   
3       9037590 2024-08-29 19:07:35.417000+00:00  [1, 0, 0, 0, 17, 6, 11]   
4      32928599 2024-12-05 22:41:01.425000+00:00    [1, 0, 0, 0, 0, 2, 0]   
...         ...                              ...                      ...   
48620  17496760 2024-06-13 11:09:38.536000+00:00    [0, 0, 0, 1, 0, 0, 0]   
48621  26349450 2024-11-12 13:30:26.387000+00:00    [0, 0, 0, 0, 0, 0, 0]   
48622  20463978 2024-09-01 20:00:41.869000+00:00    [2, 0, 0, 0, 0, 0, 0]   
48623   2293646 2024-11-16 00:46:05.321000+00:00    [0, 0, 0, 0, 0, 0, 0]   
48624  26149935 2025-02-15 18:46:02.865000+00:00    [0, 0, 0, 1, 0, 0, 0]   

             blocks_actor_vec     blocks

## Save the file